# fourierencoding

Historical experiment source from the internship project. Outputs and personal runtime metadata are cleared. This version is retained for provenance; see the root README for the maintained entry points. Colab Drive paths describe the original environment and require your own matching inputs. Source: `fourierencoding.py`. No fresh GPU execution is claimed.

In [ ]:
# -*- coding: utf-8 -*-
"""FourierEncoding.ipynb

Automatically generated by Colab.

Original file is located at
    [original private Colab link omitted]
"""

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd

CSV_FOLDER = "/content/drive/MyDrive/Fourier/ExtractedCSV"

MASTER_FILE = "/content/drive/MyDrive/Fourier/master_object_list.csv"

master_df = pd.read_csv(MASTER_FILE)

master_objects = master_df["Object"].tolist()

print(master_objects)

print(len(master_objects))

csv_files = sorted(

    glob.glob(

        os.path.join(CSV_FOLDER,"*.csv")

    )

)

print(csv_files)

scene_vectors = {}

for csv_file in csv_files:

    room_name = os.path.splitext(

        os.path.basename(csv_file)

    )[0]

    df = pd.read_csv(csv_file)

    room_vector = []

    for obj in master_objects:

        row = df[df["Object"] == obj]

        if len(row) == 0:

            room_vector.extend([0.0,0.0,0.0])

        else:

            room_vector.extend(

                [

                    float(row.iloc[0]["X"]),

                    float(row.iloc[0]["Y"]),

                    float(row.iloc[0]["Z"])

                ]

            )

    scene_vectors[room_name] = np.array(

        room_vector,

        dtype=np.float32

    )

for room in scene_vectors:

    print(room,scene_vectors[room].shape)

def fourier_encode(

    vector,

    num_frequencies=6

):

    vector = np.asarray(

        vector,

        dtype=np.float32

    )

    encoded = []

    for value in vector:

        for k in range(num_frequencies):

            freq = 2.0**k

            encoded.append(

                np.sin(

                    freq*np.pi*value

                )

            )

            encoded.append(

                np.cos(

                    freq*np.pi*value

                )

            )

    return np.array(

        encoded,

        dtype=np.float32

    )

fourier_vectors = {}

for room in scene_vectors:

    vec = fourier_encode(

        scene_vectors[room],

        num_frequencies=6

    )

    vec = vec / np.linalg.norm(vec)

    fourier_vectors[room] = vec

for room in fourier_vectors:

    print(

        room,

        fourier_vectors[room].shape,

        np.linalg.norm(

            fourier_vectors[room]

        )

    )

room_names = sorted(

    fourier_vectors.keys()

)

embedding_matrix = np.stack(

    [

        fourier_vectors[r]

        for r in room_names

    ]

)

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(

    embedding_matrix

)

similarity_df = pd.DataFrame(

    similarity_matrix,

    index=room_names,

    columns=room_names

)

similarity_df.round(4)

import matplotlib.pyplot as plt

import seaborn as sns

plt.figure(

    figsize=(8,6)

)

sns.heatmap(

    similarity_df,

    annot=True,

    cmap="viridis",

    fmt=".3f"

)

plt.title(

    "Fourier Coordinate Similarity Matrix"

)

plt.tight_layout()

plt.show()

plt.figure(figsize=(12,10))

sns.heatmap(

    similarity_df,

    annot=True,

    cmap="viridis",

    fmt=".3f",

    square=True,

    linewidths=0.5

)

plt.title(

    "Fourier Similarity Matrix"

)

plt.tight_layout()

plt.savefig(

"/content/drive/MyDrive/Fourier/Fourier_Heatmap.png",

dpi=300

)

plt.close()